# Validación experimental de agentes MCTS para Connect-4

## SECCIÓN 1: INTRODUCCIÓN Y PROTOCOLO EXPERIMENTAL

Connect-4 puede modelarse como un **juego de Markov alternante** (*Alternating Markov Game*) de dos jugadores, determinista, de suma cero y con información perfecta. Un estado $s_t$ contiene la configuración del tablero y el jugador activo; una acción $a_t$ corresponde a seleccionar una columna legal; la transición $s_{t+1}=T(s_t,a_t)$ deposita una ficha y alterna el turno. La recompensa terminal asigna victoria, derrota o empate. Aunque las transiciones sean deterministas, la gran cantidad de secuencias legales hace necesario aproximar la política de decisión mediante búsqueda selectiva.

Este estudio contrasta dos políticas del grupo frente a un oponente de control que selecciona columnas al azar:

- **Agente Base, `Fermin_MCTS`** (`groups/Fermin_MCTS/policy.py`): Monte-Carlo Tree Search (MCTS) con selección UCB1 estándar, expansión incremental y rollouts aleatorios.
- **Agente Optimizado, `Fermin_MCTS_Time&C`** (`groups/Fermin_MCTS_Time&C/policy.py`): versión con gestión dinámica del presupuesto temporal y constante de exploración dependiente de la profundidad. La configuración avanzada que se desea validar incluye además **tabla de transposición** y **reúso del árbol**.
- **Oponente de Control, `Random`** (`groups/Random/policy.py`): política aleatoria sobre columnas disponibles.

> **Nota de trazabilidad.** La inspección del archivo optimizado disponible en este repositorio confirma el tiempo dinámico y la variación de $C$ por profundidad, pero no evidencia todavía una estructura persistente de tabla de transposición ni el reúso de una raíz entre turnos. Por ello, cuando no existan resultados instrumentados en `versus/`, las curvas de este notebook se etiquetan como **datos simulados del protocolo objetivo**; sirven para diseñar y comunicar el experimento, no para afirmar que esas dos optimizaciones ya fueron medidas.

### Protocolo experimental

Se plantea un torneo de **100 partidas** para cada combinación agente-presupuesto frente a `Random`, alternando el jugador inicial para reducir sesgo por iniciativa. La variable numérica de recurso es el presupuesto máximo por turno, `TURN_TIME_LIMIT`, con valores $[0.2, 0.5, 1.0, 1.8]$ segundos. Para cada configuración se registran tasa de victorias (`win_rate`), tasa de empates (`draw_rate`), profundidad media alcanzada (`avg_depth_reached`) y simulaciones ejecutadas por segundo (`simulations_per_second`). Esta última pareja de métricas requiere instrumentación adicional al DTO estándar de partidas, pues los archivos JSON originales almacenan ganadores, empates e historiales de acciones, no estadísticas internas de búsqueda.


In [2]:
# SECCIÓN 2: SIMULACIÓN DE DATOS Y CARGA DE MÉTRICAS
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

N_GAMES = 100
TIME_LIMITS = [0.2, 0.5, 1.0, 1.8]
AGENT_VERSIONS = ["Fermin_MCTS", "Fermin_MCTS_Time&C"]
VERSUS_DIR = Path("versus")


def _budget_tag(time_limit: float) -> str:
    """Convierte 0.2 en el sufijo estable 0p2 utilizado para resultados instrumentados."""
    return str(time_limit).replace(".", "p")


def _dto_record(payload: dict, agent_name: str, time_limit: float) -> dict:
    """Extrae métricas de un JSON compatible con connect4.dtos.Match.

    `avg_depth_reached` y `simulations_per_second` son campos opcionales de una
    corrida instrumentada; el DTO base no los produce por sí mismo.
    """
    total_games = payload["player_a_wins"] + payload["player_b_wins"] + payload["draws"]
    if total_games == 0:
        raise ValueError("Un resultado experimental no puede contener cero partidas.")

    if payload["player_a"] == agent_name:
        wins = payload["player_a_wins"]
    elif payload["player_b"] == agent_name:
        wins = payload["player_b_wins"]
    else:
        raise ValueError(f"El agente {agent_name} no aparece en el archivo de resultados.")

    return {
        "version_agente": agent_name,
        "time_limit_per_turn": time_limit,
        "win_rate": wins / total_games,
        "draw_rate": payload["draws"] / total_games,
        "avg_depth_reached": payload.get("avg_depth_reached", np.nan),
        "simulations_per_second": payload.get("simulations_per_second", np.nan),
    }


def load_complete_instrumented_benchmark(versus_dir: Path = VERSUS_DIR) -> pd.DataFrame | None:
    """Carga ocho corridas contra Random si fueron exportadas con instrumentación.

    Convención esperada: `match_<agente>_vs_Random_t<budget>.json`, por ejemplo
    `match_Fermin_MCTS_vs_Random_t0p5.json`. La función retorna `None` si falta
    alguna corrida o si los JSON no incluyen métricas internas de MCTS.
    """
    records = []
    for agent_name in AGENT_VERSIONS:
        for time_limit in TIME_LIMITS:
            path = versus_dir / f"match_{agent_name}_vs_Random_t{_budget_tag(time_limit)}.json"
            if not path.exists():
                return None
            with path.open("r", encoding="utf-8") as result_file:
                payload = json.load(result_file)
            record = _dto_record(payload, agent_name, time_limit)
            if np.isnan(record["avg_depth_reached"]) or np.isnan(record["simulations_per_second"]):
                return None
            records.append(record)
    return pd.DataFrame(records)


def simulate_protocol_metrics() -> pd.DataFrame:
    """Genera resultados plausibles y reproducibles para el diseño de validación.

    Los enteros de victorias/empates representan 100 partidas por condición y
    respetan la hipótesis de que ambos MCTS superan ampliamente al control.
    """
    scenarios = {
        "Fermin_MCTS": {
            0.2: (86, 3, 5.8, 720),
            0.5: (89, 3, 7.0, 728),
            1.0: (93, 2, 8.5, 735),
            1.8: (96, 1, 9.7, 741),
        },
        "Fermin_MCTS_Time&C": {
            0.2: (93, 2, 7.7, 1045),
            0.5: (95, 2, 9.0, 1082),
            1.0: (97, 1, 10.4, 1106),
            1.8: (98, 1, 11.6, 1124),
        },
    }
    records = []
    for agent_name, measurements in scenarios.items():
        for time_limit, (wins, draws, avg_depth, simulations_per_second) in measurements.items():
            records.append({
                "version_agente": agent_name,
                "time_limit_per_turn": time_limit,
                "win_rate": wins / N_GAMES,
                "draw_rate": draws / N_GAMES,
                "avg_depth_reached": avg_depth,
                "simulations_per_second": simulations_per_second,
            })
    return pd.DataFrame(records)


def load_or_simulate_metrics(versus_dir: Path = VERSUS_DIR) -> tuple[pd.DataFrame, str]:
    observed = load_complete_instrumented_benchmark(versus_dir)
    if observed is not None:
        return observed, "observado: torneo instrumentado cargado desde versus/"
    return simulate_protocol_metrics(), "simulado: protocolo objetivo de 100 partidas por condición"


df, data_source = load_or_simulate_metrics()
df = df.sort_values(["version_agente", "time_limit_per_turn"]).reset_index(drop=True)
print(f"Fuente de métricas: {data_source}")
df


ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# SECCIÓN 3: CRITERIO 1 - Win rate en función del presupuesto por turno
CONTROL_RANDOM_WIN_RATE = 0.03
agent_palette = {
    "Fermin_MCTS": sns.color_palette("viridis", n_colors=3)[0],
    "Fermin_MCTS_Time&C": sns.color_palette("viridis", n_colors=3)[2],
}

fig, ax = plt.subplots(figsize=(12, 6.5))
sns.lineplot(
    data=df,
    x="time_limit_per_turn",
    y="win_rate",
    hue="version_agente",
    style="version_agente",
    markers=True,
    dashes=False,
    linewidth=2.8,
    markersize=10,
    palette=agent_palette,
    ax=ax,
)
ax.axhline(
    CONTROL_RANDOM_WIN_RATE,
    color="dimgray",
    linestyle="--",
    linewidth=2,
    label="Random (referencia: 3%)",
)
ax.set_title("Rendimiento frente a Random según TURN_TIME_LIMIT", weight="bold", pad=16)
ax.set_xlabel("Presupuesto por turno, TURN_TIME_LIMIT (segundos)")
ax.set_ylabel("Tasa de victorias (Win Rate)")
ax.set_xticks(TIME_LIMITS)
ax.set_ylim(0, 1.04)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(title="Política evaluada", loc="center right")
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# SECCIÓN 3: CRITERIO 1 - Victorias medias acumuladas en 100 partidas
wins_summary = (
    df.assign(victorias_en_100=lambda values: values["win_rate"] * N_GAMES)
      .groupby("version_agente", as_index=False)["victorias_en_100"]
      .mean()
      .rename(columns={"victorias_en_100": "victorias_promedio_en_100"})
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=wins_summary,
    x="version_agente",
    y="victorias_promedio_en_100",
    hue="version_agente",
    dodge=False,
    palette=agent_palette,
    ax=ax,
)
legend = ax.get_legend()
if legend is not None:
    legend.remove()
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=4, fontsize=12)
ax.axhline(85, color="firebrick", linestyle="--", linewidth=1.8, label="Umbral de 85 victorias")
ax.set_title("Victorias promedio contra Random por cada 100 partidas", weight="bold", pad=16)
ax.set_xlabel("Versión del agente")
ax.set_ylabel("Victorias promedio / 100 partidas")
ax.set_ylim(0, 105)
ax.legend(loc="lower right")
sns.despine()
plt.tight_layout()
plt.show()


## SECCIÓN 3: CRITERIO 1 - ANÁLISIS DEL AGENTE Y RENDIMIENTO DEL RECURSO

En el escenario simulado del protocolo, ambas políticas MCTS superan el 85% de victorias contra el agente aleatorio en todos los presupuestos. La versión base presenta la sensibilidad esperada al recurso: su `win_rate` aumenta de 86% con 0.2 s a 96% con 1.8 s. Esto es coherente con MCTS, pues más tiempo permite incrementar $N(s,a)$, reducir varianza en la estimación Monte-Carlo de los retornos y distinguir con mayor confianza acciones inicialmente similares.

La curva objetivo de `Fermin_MCTS_Time&C` es más estable: alcanza 93% con 0.2 s y 98% con 1.8 s. La ventaja es particularmente relevante en baja latencia, donde una política que administra el presupuesto y concentra la exploración cerca de la raíz puede dedicar las muestras a decisiones con impacto inmediato en la jugada seleccionada. Si la tabla de transposición y el reúso de árbol se incorporan efectivamente, estados ya estudiados aportarían estadísticas acumuladas sin reconstruir completamente la búsqueda, incrementando el rendimiento útil por segundo.

No obstante, **robustez estadística** no debe confundirse con una tabla sintética: con 100 partidas, una proporción cercana a 0.93 todavía tiene un error estándar binomial aproximado de 2.6 puntos porcentuales. Una validación concluyente debe ejecutar el torneo instrumentado con varias semillas, balancear el color inicial y reportar intervalos de confianza o pruebas para proporciones. Hasta contar con esos archivos, los resultados mostrados documentan la hipótesis experimental y el comportamiento esperado, no una medición definitiva del artefacto actual.


In [ ]:
# SECCIÓN 4: CRITERIO 2 - Eficiencia computacional de las simulaciones
efficiency_mean = df.groupby("version_agente", as_index=False)["simulations_per_second"].mean()

fig, ax = plt.subplots(figsize=(11, 6.5))
sns.scatterplot(
    data=df,
    x="version_agente",
    y="simulations_per_second",
    hue="time_limit_per_turn",
    palette="coolwarm",
    s=150,
    edgecolor="black",
    linewidth=0.6,
    ax=ax,
)
for index, row in efficiency_mean.iterrows():
    ax.hlines(
        row["simulations_per_second"],
        index - 0.22,
        index + 0.22,
        colors="black",
        linewidth=3,
    )
    ax.text(
        index,
        row["simulations_per_second"] + 20,
        f"media: {row['simulations_per_second']:.0f}",
        ha="center",
        fontsize=11,
        weight="bold",
    )
ax.set_title("Simulaciones por segundo: indicio del cuello de botella", weight="bold", pad=16)
ax.set_xlabel("Versión del agente")
ax.set_ylabel("Simulaciones por segundo")
ax.legend(title="TURN_TIME_LIMIT (s)", bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine()
plt.tight_layout()
plt.show()


## SECCIÓN 4: CRITERIO 2 - RELACIÓN CAUSA-EFECTO

El MCTS base crea una raíz nueva en cada llamada a `act()` y estima cada trayectoria desde nodos independientes. En Connect-4, distintas secuencias de columnas pueden conducir a configuraciones equivalentes o reutilizables para evaluación; si cada ocurrencia vuelve a expandirse y simularse, el coste de `transition()`, comprobación terminal y rollout se repite. El efecto observable esperado es una menor tasa de simulaciones efectivas, especialmente cuando el presupuesto es corto y cada cálculo redundante desplaza una muestra potencialmente informativa de la decisión en la raíz.

La arquitectura optimizada objetivo aborda tres causas complementarias:

- **Tabla de transposición.** Una clave inmutable del tablero y jugador activo permite consultar estadísticas ya acumuladas. La consulta de una tabla hash tiene coste esperado amortizado $O(1)$; esto no vuelve $O(1)$ a MCTS completo, pero evita expansiones y evaluaciones redundantes. Conceptualmente, los caminos que convergen en un estado comparten información, aproximando la estructura de búsqueda a un grafo dirigido acíclico en lugar de árboles aislados.
- **Reúso del árbol entre turnos.** Después de ejecutar una acción y observar la respuesta del rival, el subárbol compatible puede promoverse como nueva raíz. Así no se descartan visitas ni valores $Q/N$ obtenidos en turnos anteriores, aumentando la información disponible antes de consumir el presupuesto nuevo.
- **Constante `C` variable por profundidad.** Una exploración alta cerca de la raíz compara alternativas que determinan la acción real; al disminuir `C` en niveles profundos, la búsqueda privilegia ramas prometedoras y reduce exploraciones periféricas de menor relevancia para la decisión inmediata.

**Limitación de validez interna.** En el código actualmente disponible, `Fermin_MCTS_Time&C` implementa tiempo dinámico y `C` variable, pero aún construye `root = Node(...)` dentro de cada `act()` y no declara una tabla de transposición persistente. Por tanto, la diferencia de `simulations_per_second` dibujada arriba debe probarse nuevamente después de implementar e instrumentar explícitamente esos mecanismos; no puede atribuirse causalmente al archivo actual solo a partir de la simulación.


## SECCIÓN 4: CRITERIO 2 - FUTURAS MEJORAS

Una mejora concreta es incorporar un **libro de aperturas persistente** para los primeros tres turnos. Se almacenaría en disco una tabla versionada que mapee `(tablero_canonico, jugador_activo)` a una acción recomendada y sus estadísticas de validación. La canonicalización por reflexión horizontal reduce estados duplicados, y una consulta hash permitiría decidir aperturas conocidas con tiempo de búsqueda prácticamente nulo, reservando el presupuesto de MCTS para posiciones de medio juego en las que la incertidumbre es mayor.

La propuesta debe evaluarse mediante una ablación controlada: `MCTS base`, `Time&C`, `Time&C + transposición/reúso` y `Time&C + transposición/reúso + libro de aperturas`, todos frente a `Random` y a oponentes no aleatorios, con las mismas semillas y distribución de colores. Además del `win_rate`, debe registrarse tiempo real por jugada, tasa de aciertos del libro y fracción de partidas que abandona el libro sin encontrar un estado conocido; así se comprueba que el ahorro temporal no introduce una política frágil ante aperturas adversarias.
